# Calibration jig demo

CO2Dot, LI-6800, IR lamp and Peltier in a few lines each. The code behind every call is in
`calibration_utils.py`, in this folder.

Needs `pip install pyserial paho-mqtt` (and `requests` for the optional openJII streaming).
**Always run the last cell when you are done:** it switches the lamp and the Peltier off.

In [ ]:
import calibration_utils as cu

In [ ]:
# Find every device. A missing one prints "cannot find ..." and is skipped; the others keep working.
jig = cu.connect()
dot, lamp, peltier, li = jig.dot, jig.lamp, jig.peltier, jig.li
stream = cu.openjii(None, jig)      # optional: replace None by an openJII experiment id to send data there

In [ ]:
# CO2Dot settings in one line: LED current (mA), integration (atime, astep) and gain.
# Gas sensing is off: cu.connect() switched it off (dot.gas_heater(True) turns it back on).
print(dot.setup(led_ma=4, atime=100, astep=999, gain=5))

# One measurement: LED flash (lit minus dark, 10 channels) + board temperature, humidity, pressure.
m = dot.measure()
print(m)

In [ ]:
# Board temperature: the IR lamp heats, the Peltier cools.
# Limits: Peltier 7 V / 2 A, lamp step 16 (of 25), everything off above 40 C.
temp = cu.TemperatureControl(jig, v_max=7.0, i_max=2.0, lamp_max=16, t_max=40.0)

# kp: reaction to the error now (per C) | ki: removes the remaining offset over time (per C per s) | kd: damps fast changes
# kp_heat, ki_heat: the same two gains while the lamp is heating (the lamp side is slower, so they are smaller)
temp.pid(kp=0.15, ki=0.0006, kd=0.0, kp_heat=0.03, ki_heat=0.0002)

temp.set(27.0)                      # target in C; the loop keeps holding it after this cell

# Wait until the temperature has stayed within 1 % of the target for 2 minutes (gives up after 60 minutes).
cu.wait_stable(temp.read, 27.0, tol_pct=1, hold_s=120, timeout_s=3600)
print(temp.status())

In [ ]:
# LI-6800: everything it returns, then a subset.
for k, v in sorted(li.read().items()):
    print(f"{k:<14} {v}")
print({k: li.get(k) for k in ("CO2_s", "CO2_r", "H2O_s", "Tchamber", "Txchg", "Tleaf", "RHcham", "Flow")})

# Set some parameters (this drives the instrument immediately) ...
li.set(co2_s=400, flow=400, tair=22, rh_air=40)

# ... and wait until CO2 has stayed within 5 ppm of the target for 30 s (gives up after 10 minutes).
cu.wait_stable(lambda: li.get("CO2_s"), 400, tol=5, hold_s=30, timeout_s=600)

In [ ]:
# Optional: one full record (CO2Dot + LI-6800 + lamp / Peltier state) to the openJII experiment of cell 2.
if stream:
    stream.send(jig.snapshot())

In [ ]:
# Always finish here: lamp and Peltier off (checked);
# LI-6800: CO2 and H2O control off, flow left on, block at 25 C.
jig.close()